### **Extraer el `titulo`, `reseña` y `enlace de la pelicula` de cada pelicula que comience con X que se encuentre en todas las páginas y exportarlos a un archivo `JSON`**

Vamos a trabajar en el enlace:

https://subslikescript.com/movies_letter-X

#### **Crear la `crawl spider`**

Ahora vamos a crear un nuevo spider utilizando una crawl template. Así que escribimos "**`scrapy genspider -t crawl`**" y luego escribimos el nombre de la spider que queremos crear. 

<center><img src="https://i.postimg.cc/QCr4XsM1/ws-452.png"></center>

Luego, debemos editar el archivo `transcripcion_dos.py` que se crea, con el código que se muestra abajo.

#### **Paso a paso**

Vamos a inspeccionar el sitio:

<center><img src="https://i.postimg.cc/Dz5cbH23/ws-142.png"></center>

Localizamos el boton de la paginacion superior de "pagina siguiente" (existe un segundo boton que se encuntra en la paginacion inferior):

<center><img src="https://i.postimg.cc/Y0psYYFh/ws-153.png"></center>

Vamos a localizar los enlaces a cada una de las peliculas de la pagina:

<center><img src="https://i.postimg.cc/3wdJZF9d/ws-448.png"></center>

Y vamos a escoger cualquier pelicula para inspeccionar su página:

<center><img src="https://i.postimg.cc/zBBFw7Bb/ws-143.png"></center>

Localizamos la caja que encierre todo el contenido de la página:

<center><img src="https://i.postimg.cc/BbP5kv2J/ws-144.png"></center>

Localizamos su `titulo`:

<center><img src="https://i.postimg.cc/mZP0BNwg/ws-145.png"></center>

Localizamos su `introducción`:

<center><img src="https://i.postimg.cc/nM8KJt1B/ws-146.png"></center>

Y finalmente, localizamos su `transcripción`:

<center><img src="https://i.postimg.cc/3RcCtG4L/ws-147.png"></center>

#### **Código**

La razón por la que puedo eliminar el "**`callback`**" en esta segunda regla es porque ya enviamos el callback igual a "**`parse_item`**" en el objeto de la primera regla. Así que cuando visitemos cada página siguiente, el "`parse_item`" será llamado por lo que no hay necesidad de codificar de nuevo en la segunda regla. Por otro lado, podemos omitir el argumento "**`follow`**" porque no estamos especificando el argumento "**`callback`**", así que si no hay argumento "`callback`" no hay necesidad del argumento "**`follow`**". Otra cosa que necesitas saber es que el orden en las reglas importa. Así que si la segunda regla estuviese en la primera posición, no scrapearemos la primera página.

Con esto scrapearemos **`TODAS LAS PÁGINAS`** de las peliculas que comiencen con X:

In [ ]:
import scrapy
from scrapy.linkextractors import LinkExtractor
from scrapy.spiders import CrawlSpider, Rule


class TranscripcionDosSpider(CrawlSpider):
    name = "transcripcion_dos"
    allowed_domains = ["subslikescript.com"]
    #start_urls = ["https://subslikescript.com/movies_letter-X"]

    # Establecer una variable user agent
    user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'

    # Editar el user agent en la request enviada
    def start_requests(self):
        yield scrapy.Request(url='https://subslikescript.com/movies_letter-X', headers={'user-agent':self.user_agent})

    # Establecer reglas para el crawler
    # Así que seguiremos sólo los enlaces de los elementos que tengan este XPath (restrict_xpaths).
    # Además, como se puede ver, no he añadido el "/@href" (//ul[@class='scripts-list']/a/@href) esto es porque el 
    # objeto "LinkExtractor" buscará automáticamente el atributo "href", por lo que tenemos que omitirlo.
    rules = (
        Rule(LinkExtractor(restrict_xpaths=(["//ul[@class='scripts-list']/a"])), callback='parse_item', follow=True, process_request='set_user_agent'),
        Rule(LinkExtractor(restrict_xpaths=("(//a[@rel='next'])[1]"))),
    )

    # Configuración del user agent
    def set_user_agent(self, request, spider):
        request.headers['User-Agent'] = self.user_agent
        return request

    def parse_item(self, response):
        # Obtener la caja del article que contiene los datos que queremos (title, plot, etc)
        article = response.xpath("//article[@class='main-article']")

        # Extraer los datos que queremos y luego devolverlos
        yield {
            'titulo': article.xpath("./h1/text()").get(),
            'reseña': article.xpath("./p/text()").get(),
            'enlace': response.url,
        }

#### **Ejecución**

Nos posicionamos en la ruta correcta y ejecutamos nuestra Spider:

In [ ]:
scrapy crawl transcripcion_dos

<center><img src="https://i.postimg.cc/Kjr9jVKV/ws-453.png"></center>

Si nos fijamos bien, solo nos devolvió el scrapping de 19 elementos, es decir, de la primera página. Me fije que aparecia el mensaje:

- **`[scrapy.core.engine] DEBUG: Crawled (429)`** y
- **`[scrapy.downloadermiddlewares.retry] ERROR: Gave up retrying <GET url> (failed 3 times): 429 Unknown Status`**

<center><img src="https://i.postimg.cc/ZK472T2h/ws-454.png"></center>

**`Código de estado HTTP 429`**: Demasiadas peticiones significa que el servidor que estás crawleando te está limitando la velocidad. Intenta disminuir el número de requests enviadas en paralelo, y/o introducir un delay entre las requests.

Por tanto, para solucionar este problema, desde el archivo `settings.py` modificamos el valor a `DOWNLOAD_DELAY = 3`, es decir, aplicar un delay de 3 segundos entre cada petición:

In [ ]:
DOWNLOAD_DELAY = 3

Podriamos exportar toda esta data en un archivo `JSON` escribiendo el siguiente comando:

In [ ]:
scrapy crawl transcripcion_dos -o transcripcion_dos.json